# 第12回 論文のコードを読む（解答例・教員用）


---


## 前回からの接続


前回は、k-meansを自分で書き、scikit-learnの`KMeans`の中で何がくり返されているかを確かめました。中身を知っている道具なら、ほかの人が書いたコードの中に出てきても読めます。

今日は、研究者が論文といっしょに公開したコードを読みます。題材は、ハッブル宇宙望遠鏡が撮った銀河の画像に、主成分分析（principal component analysis）とクラスタリング（clustering）をかけた論文です。


---


## 今日の分析目標


あなたは天文学の研究室に入ったばかりの新人です。論文の主張を、著者が公開したコードを読んで動かし、確かめられるかに答える。

論文は「銀河の形は、12個ほどの固有銀河の足し算で表せる」と主張しています。著者のノートの地図を作り、読み込み・前処理・主成分分析・外れ値（outlier）・クラスタリングの順に、著者のセルを読んで動かします。

もう1つのねらいは、研究者のコードを、関数を作るブロックと、それを使う処理のブロックに分けて読み、動かせるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、天文学の研究室に入ったばかりの新人です。先輩から、銀河の形を調べた論文を1本渡され、「この論文のコードを動かして、主張が本当か確かめておいて」と頼まれました。

コードは著者が書いたもので、説明はほとんどありません。どこから読み、どこを動かせば、論文の主張を確かめられるでしょうか。


### 手元のデータ

論文は、Uzeirbegovic, Geach, Kaviraj（2020）"Eigengalaxies: describing galaxy morphology using principal components in image space"（MNRAS 498(3): 4021–4032）です。著者は、コードとデータをGitHubに公開しています（https://github.com/emiruz/eigengalaxies）。

このノートでは、リポジトリから著者のノートと、銀河1,222個ぶんの画像を取ってきます。1個の銀河は、縦30×横30画素で、3つの波長の帯を持つ画像です。答えの列はありません。

著者のコードは、このリポジトリのノート（`Eigengalaxies.ipynb`）のセルを、番号を添えてそのまま写しています。


### 最後に出す答え

固有銀河いくつで銀河の形の何%を表せるかを、論文の数と比べた結果です。あわせて、著者のコードを再現するときに気をつける所です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 論文に付いたノートを、見出しで地図にし、関数を作るブロックと処理のブロックに分けて読める
- `.ipynb`がJSONという形式のファイルで、中身を辞書とリストとして読めることを説明できる
- `glob`でたくさんのファイルを集め、`.npy`とPNGを、形式に合った方法で読める
- 前処理（明るさのクリップ、向きをそろえる）→ 主成分分析 → 外れ値 → クラスタリング、の各段が何をしているかを言える
- 他人のコードを再現するときに、ファイルの並び順・乱数（random number）のシード・関数が渡した配列を書きかえるか、に気づける
- コードの見出しや表示と、中身が合っているかを確かめられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import os
import json
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 論文のコードとデータを、取ってくる

- 役割：著者が公開したリポジトリから、ノートと、銀河1,222個ぶんの画像だけを取ってくる
- 入力：リポジトリの場所（URL）と、取ってくるファイルの名前の型
- 出力：`eigengalaxies`というフォルダの中身

リポジトリ全体は400MB以上あるので、`git`の機能で必要なファイルだけを取ってきます（数秒・約40MB）。

- `!`で始まる行は、Pythonではなく、計算機に直接出す命令です
- `os.chdir`は、作業の場所（フォルダ）を移す関数です。著者のコードは、このフォルダの中で動く前提で書かれているので、移しておきます


In [ ]:
WORK = '/content'   # Colab の作業場所
URL = 'https://github.com/emiruz/eigengalaxies.git'
PICK = ("'/Eigengalaxies.ipynb' '/CANDELS/GDS_[12]???.npy' "
        "'/GDS_RGBs/GDS_0[12]???.png'")      # 取ってくるファイルの型
os.chdir(WORK)
if not os.path.exists('eigengalaxies'):     # まだ取ってきていなければ
    !git clone -q --depth 1 --filter=blob:none --sparse $URL
    !cd eigengalaxies && git sparse-checkout set --no-cone {PICK}
os.chdir('eigengalaxies')                   # ここを作業の場所にする
print(sorted(os.listdir('.')))


`CANDELS`（銀河の画像の数値）、`GDS_RGBs`（カラーの縮小画像）、`Eigengalaxies.ipynb`（著者のノート）がそろいました。


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、他人の解析の①〜③を、コードから読み解きます。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の6つの小さな問いに答えていきます。

1. 著者のノートは、どこから読むか？（1節）
2. データは、どう読み込まれているか？（2節）
3. 主成分分析の前に、何をしているか？（3節）
4. 何個の固有銀河で表せるか？（4節）
5. めずらしい銀河は、どれか？（5節）
6. 似た銀河は、どう集まるか？（6節）


---


## 1. 著者のノートは、どこから読むか？


3つのファイルがそろいました。では、著者のノートのどこから読めばよいでしょうか。他人のコードは、いきなり1行目から読みません。まず、どんなまとまり（ブロック）がどの順に並んでいるかの地図を作ります。


### ノートは、JSONという形式

ノート（`.ipynb`）は、JSONという形式の文字のファイルです。JSONは、辞書（`{名前: 値}`）とリスト（`[…]`）を入れ子にして、データを書く形式です。`json.load`で読むと、Pythonの辞書とリストになります。`open`は、ファイルを開く関数です。


### お手本：著者のノートを、JSONとして読む

- 役割：著者のノートを開き、中身がどんな形で入っているかを見る
- 入力：`Eigengalaxies.ipynb`
- 出力：辞書のキー、セルの数、2番目のセルの中身


In [ ]:
nb = json.load(open('Eigengalaxies.ipynb'))   # ノートの中身は JSON
print(list(nb.keys()))
print(len(nb['cells']), 'セル')
print(nb['cells'][2])


辞書のキーに`'cells'`があり、そこにセルが35個、リストで入っています。1つのセルも辞書で、`'cell_type'`（`'markdown'`か`'code'`）と、`'source'`（中身の文字を、行ごとに分けたリスト）を持っています。セルの番号は、0から数えます。


### 変えてみる：見出しだけを並べる

- 役割：文章のセル（markdown）だけを取り出し、番号と先頭の40字を表示する
- 入力：`nb`
- 出力：見出しの一覧

`enumerate`は、番号と中身を組にして取り出します。`''.join(…)`は、行ごとに分かれた文字のリストを、1つの文字につなげます。`.strip()`は、前後の空白や改行を取り除きます。


In [ ]:
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'markdown':
        print(i, ''.join(cell['source']).strip()[:40])


見出しを並べると、ノートの地図が見えます。

| セル | 見出し | 今日 |
|---|---|---|
| 1・3 | Common imports／Constants（道具と定数） | 使う |
| 5 | Data Processing Functions（前処理の関数） | 使う（2・3節） |
| 7 | Image Generation（画像を並べる関数） | 使う |
| 9・11 | Data／PCA | 使う（2〜4節） |
| 14・18 | Checks／Sample size（主成分分析を何百回もくり返す確認） | 重いので飛ばす |
| 16 | Outlier Detection（外れ値） | 使う（5節） |
| 20・22・24 | 累積寄与率（cumulative proportion of variance explained）／再構成の誤差／The Eigengalaxies | 24を使う（4節） |
| 26 | Imputing Values（欠けた値の補い） | 別の道具が要るので飛ばす |
| 33・34 | Affinity Propagation Clustering | 使う（6節） |


### TODO①：コードのセルの1行目を並べる

お手本をまねて、今度はコードのセル（`'code'`）だけを取り出し、番号と1行目を表示してください。1行目だけで、どのセルが何をしているかが、どこまで分かるでしょうか。


In [ ]:
# TODO: コードのセルの番号と、1行目を表示する

# 解答例①：コードのセルの1行目
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'code':
        print(i, cell['source'][0].strip())


コードのセルは19個あります。`def rescale(a):`（5）、`import matplotlib.image as mpimg`（7）、`raw_data,indexes=get_data()`（9）、`pca=PCA(.96).fit(norm_data)`（11）のように、1行目だけでも、関数を作るブロック（5・7）と、それを使って処理するブロック（9・11）が見分けられます。


### 道具と定数のセル

著者のセル1（道具）とセル3（定数）を、そのまま写して実行します。ここから先、著者のセルは、番号を添えてそのまま写したものです。


In [ ]:
# 著者のセル1をそのまま写したもの
import re,os
import numpy as np
import pandas as pd
from glob import glob
import sklearn
from sklearn.decomposition import PCA
from mpl_toolkits.axes_grid1 import ImageGrid
import matplotlib.pyplot as plt
from skimage.transform import rotate
from sklearn import tree
from sklearn.metrics import mean_squared_error
import matplotlib.image as mpimg
from matplotlib.ticker import MultipleLocator,FormatStrFormatter,AutoMinorLocator


In [ ]:
# 著者のセル3をそのまま写したもの
SHP=(30,30,3)
BAND_ORDER=[2,0,1]
BANDS=["F814W","F125W","F160W"]


In [ ]:
print(SHP, BAND_ORDER, BANDS)


`SHP=(30,30,3)`は、1つの銀河の画像の大きさ（縦30×横30画素×3つの帯）です。`BANDS`は、ハッブル宇宙望遠鏡が撮った3つの波長の帯の名前です。


---


## 2. データは、どう読み込まれているか？


地図ができ、道具もそろいました。では、データはどうやって読み込まれているのでしょうか。データは、2つの形式で入っています。

- `CANDELS/GDS_1000.npy`など：numpyの配列を、そのまま保存した形式（.npy）。`np.load`で読む
- `GDS_RGBs/GDS_01000.png`など：ふつうの画像の形式（PNG）。`plt.imread`で読む


### お手本：1つの銀河を、2つの形式で読む

- 役割：同じ銀河の .npyとPNGを読み、形・型・値の範囲を比べる
- 入力：2つのファイル名
- 出力：配列の形・型・最小と最大


In [ ]:
x = np.load('CANDELS/GDS_1000.npy', allow_pickle=True)
im = plt.imread('GDS_RGBs/GDS_01000.png')
print(x.shape, x.dtype, x.min().round(3), x.max().round(3))
print(im.shape, im.dtype, im.min(), im.max())


どちらも30×30×3の配列ですが、中身が違います。.npyは望遠鏡が測った明るさの数値そのもので、マイナスの値もあります。PNGは、人が見るために0〜1に直したカラー画像です。解析には .npy、目で見るにはPNGを使います。


### 変えてみる：たくさんのファイルを、globで集める

- 役割：`CANDELS`の中の .npyのファイルの名前を、すべて集める
- 入力：ファイルの名前の型
- 出力：ファイルの数と、先頭の3つ

`glob('CANDELS/*.npy')`は、`*`の所に何が入ってもよい、という型に合うファイルの名前を、すべてリストにして返します。


In [ ]:
files = glob('CANDELS/*.npy')
print(len(files))
print(files[:3])


1,222個です。ただし、並びは`GDS_1000`、`GDS_1001`… の順とは限りません。`glob`が返す順番は、計算機やファイルの置かれ方によって変わることがあります。覚えておいてください。


### 前処理の関数のセル

著者のセル5には、関数が5つ入っています。長いセルですが、いまは実行するだけでかまいません。今日の読み込みに使うのは`get_data`、前処理に使うのは`normalise_all`です。


In [ ]:
# 著者のセル5をそのまま写したもの
def rescale(a):
    a=a-a.min()
    a=a/a.max()
    return a

def get_angle(d):
    fit=PCA(n_components=2).fit(
        np.argwhere(d>=np.quantile(d,.75)))
    return np.arctan2(*fit.components_[0])

def rotation(d):
    fl=np.apply_along_axis(max,2,d.reshape(SHP))
    angle=get_angle(fl)
    for i in range(0,d.shape[2]):
        d[:,:,i]=rotate(d[:,:,i],
                        angle/np.pi*180-90,
                        clip=True)
    return d

def normalise_all(a,lower_q,upper_q):
    a=a.copy()
    x=SHP[0]*SHP[1]
    L,U=np.quantile(a,[lower_q,upper_q])
    a=np.clip(a,L,U)
    for i in range(0,len(a)):
        d=a[i,:].reshape(SHP)
        d=rotation(d)
        a[i,:]=d.flatten()
    return a

def get_data():
    L=[]
    def f(x): return re.search("[^/]+(?=.npy)",x)[0]
    fns=[x for x in map(f,glob("CANDELS/*.npy"))]
    skip=[]
    for fn in fns:
        x=np.load("CANDELS/%s.npy" % fn,allow_pickle=True)
        skip.append(any([x[:,:,i].max()==0 for i in range(0,SHP[2])]))
        L.append(x.flatten())
    a=np.row_stack(L)
    I=(a.std(axis=1)!=0) & ~(np.array(skip))
    return a[I],np.array(fns)[I]

def get_catalog():
    def f(x): return re.search("[^/]+(?=.npy)",x)[0]
    fns=[x for x in map(f , glob("CANDELS/*.npy"))]
    cat=pd.read_csv("CANDELS/catalog.csv")
    cat=cat[cat.ID.isin(fns)]
    cat=cat.sort_values("ID")
    cat=cat.reset_index(drop=True)
    return cat


### get_dataを、かたまりで読む

1. 名前を集める：`glob`で .npyのファイルを集め、`re.search`で`GDS_1000`のような名前の部分を取り出す（`re`は、文字の型で探す道具）
2. 1つずつ読む：`for`で、`np.load`してから`flatten()`で2,700個（30×30×3）の横一列にし、リスト`L`にためる。3つの帯のどれかが真っ暗（最大が0）なら、`skip`にTrueを記録する
3. 表にして、外す：`np.row_stack(L)`で縦に積んで表にし、明るさが一様（uniform）（標準偏差が0）なものと、`skip`がTrueのものを外す
4. 返す：表と、名前の並び


### TODO②：外す条件を、1つの銀河で確かめる

`get_data`が外す2つの条件を、お手本で読んだ`x`（`GDS_1000`）で確かめてください。① 3つの帯それぞれの最大が0か、② 全体の標準偏差（standard deviation）が0でないか。


In [ ]:
# TODO: x の3つの帯の最大が0かどうかと、標準偏差が0でないかを表示する

# 解答例②：外す条件を1つの銀河で
print([x[:, :, i].max() == 0 for i in range(3)])
print(x.std() != 0)


結果は、3つとも`False`（`np.False_`と表示されます）と、`True`です。どの帯も真っ暗ではなく、明るさにばらつきもあるので、この銀河は外されません。著者は、撮影の失敗で一部の帯が写っていない画像を、こうして機械的に除いていたのです。


### お手本：読み込んで、並べ方をそろえる

- 役割：著者のセル9の1行目（`get_data`を呼ぶ）を実行し、そのあと、こちらで1段足して名前の順に並べ直す
- 入力：`CANDELS`のファイル
- 出力：表の大きさと、先頭の3つの名前

`glob`の順番が計算機で変わると、このあとの結果の並びも変わってしまいます。そこで、名前の順にそろえます。`np.argsort`は、並べ替えたときの順番（番号）を返します。


In [ ]:
raw_data,indexes=get_data()             # 著者のセル9の1行目
order = np.argsort(indexes)            # ここから2行は、こちらで足した
raw_data, indexes = raw_data[order], indexes[order]
print(raw_data.shape, indexes[:3])


1,222行 × 2,700列の表になりました。1行が1つの銀河です。1,222個のファイルを読んで、1,222行が残ったので、外された銀河はありませんでした。


---


## 3. 主成分分析の前に、何をしているか？


1,222個の銀河を、表に読み込めました。では、主成分分析の前に何をしているのでしょうか。著者のセル9の2行目は、`normalise_all(raw_data, 0, .99)`を呼んでいます。この関数は、2つの前処理をします。

- 明るさのクリップ：表全体の明るさの0%点〜99%点の外を、その値に切りそろえる（`np.clip`）。ごく一部の、とても明るい画素に引っぱられないため
- 向きをそろえる：銀河ごとに、長く伸びている向きを求め、同じ向きに回す（`rotation`）。同じ形なのに向きだけ違う銀河を、違う形と数えないため


### 画像を並べる関数のセル

前と後を見比べるために、著者のセル7（画像を並べる関数）を実行します。`make_image`は名前に合うPNGを並べ、`make_image_pre`は表の行を画像にして並べます。


In [ ]:
# 著者のセル7をそのまま写したもの
import matplotlib.image as mpimg

def make_image(I,figsize):
    L=[]
    for i in I.flatten():
        id=re.search("[0-9]+",i)[0].zfill(5)
        fn="GDS_RGBs/GDS_%s.png" % id
        if os.path.exists(fn): L.append(mpimg.imread(fn))
    fig=plt.figure(figsize=figsize)
    grid=ImageGrid(fig,111,nrows_ncols=figsize,axes_pad=0)
    for ax, im in zip(grid,L):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.imshow(im)
    plt.tight_layout()
    return plt

from mpl_toolkits.axes_grid1 import ImageGrid

def make_image_pre(X,figsize,title=None):
    L=[]
    for i in range(0,len(X)):
        L.append(X[i,:].reshape(SHP))
    fig=plt.figure(figsize=figsize)
    if not title is None: fig.suptitle(title, fontsize=14)
    grid=ImageGrid(fig,111,nrows_ncols=figsize,axes_pad=0)
    for ax, im in zip(grid,L):
        r=im[:,:,0]; r-=r.min()
        if r.max()!=0: r/=r.max()
        g=im[:,:,1]; g-=g.min()
        if g.max()!=0: g/=g.max()
        b=im[:,:,2]; b-=b.min()
        if b.max()!=0: b/=b.max()
        ax.set_xticks([])
        ax.set_yticks([])
        ax.imshow(np.dstack([g,r,b]))
    #plt.tight_layout(h_pad=10)
    return plt


### お手本：前処理の前と後を並べる

- 役割：著者のセル9の2行目（前処理）を実行し、前と後の9個を並べて見る
- 入力：`raw_data`
- 出力：前処理した表`norm_data`と、2枚の画像の並び

`.copy()`（写し）を渡しているのには、わけがあります。セル7を読むと、`make_image_pre`は描く前に`r-=r.min()`のように、受け取った配列の中身をその場で書きかえています。`raw_data[:9]`のように表の一部をそのまま渡すと、元の表の値まで書きかわり、あとの主成分分析の結果が変わってしまいます。


In [ ]:
norm_data=normalise_all(raw_data,0,.99)   # 著者のセル9の2行目
make_image_pre(raw_data[:9].copy(), (3, 3), '前')   # 写しを渡す
make_image_pre(norm_data[:9].copy(), (3, 3), '後')
plt.show()


前処理の後では、伸びた銀河の多くが、縦の向きに回されています。丸い銀河は、向きがはっきりしないので、見た目はほとんど変わりません。


### 向きは、主成分分析で測る

`get_angle`の中身を読むと、ここにも主成分分析が出てきます。

- `np.argwhere(d >= np.quantile(d, .75))`：明るさが上位25%の画素の座標（縦・横の番号）を集める
- その座標の点の集まりに主成分分析をかける。第1主成分（principal component）は、点がいちばん長く散らばる向き、つまり銀河が伸びている向き
- `np.arctan2`で、その向きを角度に直す

主成分分析は、データがいちばん散らばる向きを探す道具でした（第9回）。ここでは、画素の座標に使って、銀河の向きを測っています。


### 変えてみる：1つの銀河の角度を測る

- 役割：細長い銀河`GDS_1601`の、伸びている向きを度で測る
- 入力：`GDS_1601`の .npy
- 出力：角度（度）

`rotation`と同じく、画素ごとに3つの帯のいちばん明るい値を取った画像`fl`を作ってから、`get_angle`を呼びます。`np.degrees`は、角度の単位を度に直します。


In [ ]:
gal = np.load('CANDELS/GDS_1601.npy', allow_pickle=True)
fl = np.apply_along_axis(max, 2, gal)
print(round(np.degrees(get_angle(fl)), 1))


`GDS_1601`は、46.3度の向きに伸びていました。`rotation`は、この角度に合わせて回し、縦（90度）の向きにそろえます。


### 変えてみる：全部の銀河で、前と後の角度を比べる

- 役割：1,222個すべての銀河の角度を、前処理の前と後で測り、真ん中の値を比べる
- 入力：`raw_data`と`norm_data`
- 出力：2つの中央値

1行（1つの銀河）ずつ角度を測る関数`angle`を作り、`[ ]`の中の`for`でリストに集めます。角度は向きなので、絶対値（`np.abs`）で見ます。


In [ ]:
def angle(row):                      # 1つの銀河の、伸びた向き（度）
    fl = np.apply_along_axis(max, 2, row.reshape(SHP))
    return np.degrees(get_angle(fl))

before = np.abs([angle(r) for r in raw_data])
after = np.abs([angle(r) for r in norm_data])
print(np.median(before).round(1), np.median(after).round(1))


前処理の前は、角度の中央値が40度台で、向きはばらばらです。前処理の後は、中央値が90度近く（縦）に集まります。著者の前処理は、ねらいどおり向きをそろえていました。


---


## 4. 何個の固有銀河で表せるか？


向きのそろった表ができました。いよいよ、論文の主張を確かめます。1,222個 × 2,700列の表に、主成分分析をかけます。第9回の顔と同じく、主成分は2,700個の数の並びなので、30×30×3に戻せば画像として見られます。これを固有銀河（eigengalaxy）と呼びます。


### 主成分分析のセル

- 役割：累積寄与率が96%に届くまでの主成分を求め、各銀河の得点と、ありそうさを出す
- 入力：`norm_data`
- 出力：`pca`・`scores`（得点）・`lik`（5節で使う）と、主成分の数

`PCA(.96)`のように、`n_components`に0〜1の数を渡すと、累積寄与率がこの割合に届くまでの数を、自動で選びます。


In [ ]:
# 著者のセル11をそのまま写したもの
pca=PCA(.96).fit(norm_data)
scores=pca.transform(norm_data)
lik=pca.score_samples(norm_data)
print(pca.n_components_)


10個の固有銀河で、1,222個の銀河の散らばりの96%を表せました。論文は、約1万個の銀河で約12個としており、近い値です。


In [ ]:
print(pca.explained_variance_ratio_.sum().round(3))
print(pca.explained_variance_ratio_[:3].round(3))


10個の寄与率（proportion of variance explained）を足すと0.96、第1主成分だけで0.763です。


### 固有銀河を描くセル

著者のセル24は、10個の固有銀河を、3つの帯ごとに画像にして、縦に並べます。


In [ ]:
# 著者のセル24をそのまま写したもの
L=[]
N=pca.n_components_
for i in range(0,N):
    comp=pca.components_[i]
    comp=comp.reshape(SHP)
    mn=comp.min();mx=comp.max()
    for j in BAND_ORDER: L.append((mn,mx,comp[:,:,j]))
fig=plt.figure(figsize=(N,3*5))
grid=ImageGrid(fig,111,nrows_ncols=(N,3),axes_pad=0)
for i,b in enumerate(BANDS): grid[i].set_title(b)
for ax, tup in zip(grid,L):
    ax.set_xticks([])
    ax.set_yticks([])
    mn,mx,im=tup
    ax.imshow(im,cmap="gray", aspect="auto",vmin=mn,vmax=mx)
#plt.savefig("components.png")
plt.show()


1行目（第1主成分）は、大きく広がった明るさで、銀河全体の明るさと大きさにあたります。2行目は、縦に伸びた明るい中心です。向きを縦にそろえたので、縦に伸びているかどうかが、2番目に大きな違いとして現れました。後ろの行ほど、細かい形の違いです。


### TODO③：9割を表すには、何個か

`PCA()`（すべての主成分）を`norm_data`にかけて累積寄与率を求め、初めて0.9以上になる主成分の数を求めてください（第9回と同じ書き方です）。


In [ ]:
# TODO: 累積寄与率が初めて 0.9 以上になる主成分の数を求める

# 解答例③：9割を表す主成分の数
p = PCA().fit(norm_data)
cum = np.cumsum(p.explained_variance_ratio_)
print(np.argmax(cum >= 0.9) + 1)


3個です。3個で9割、10個で96%を表せます。銀河の形のほとんどは、ごく少ない固有銀河で表せるという、論文の主張どおりです。


### お手本：組み立て直して、確かめる

- 役割：固有銀河2個・10個・50個で、先頭の9個の銀河を組み立て直して並べる
- 入力：`norm_data`
- 出力：3枚の画像の並び

`inverse_transform`は、得点から元の2,700列の形に戻すメソッドです（第9回の再構成と同じ計算）。


In [ ]:
for k in [2, 10, 50]:
    p = PCA(k).fit(norm_data)
    rec = p.inverse_transform(p.transform(norm_data[:9]))
    make_image_pre(rec, (3, 3), f'{k}個')
plt.show()


2個では、ぼんやりした楕円です。10個で、銀河ごとの伸び方や明るさの片寄りまで戻ります。50個では、前処理の後の画像とほとんど同じです。


---


## 5. めずらしい銀河は、どれか？


10個の固有銀河で、銀河の形がほぼ戻りました。では、うまく表せない銀河はあるでしょうか。著者のセル11で作った`lik`を使います。


### ありそうさ：score_samples

`pca.score_samples`は、各銀河が、この主成分分析のつくる「ふつうの散らばり」にどれくらいありそうかを、数にします（対数尤度。大きいほどありそう）。いちばん小さい銀河は、固有銀河の足し算ではうまく表せない、めずらしい形の銀河です。


### TODO④：いちばんめずらしい5個

`lik`の小さい順に並べる番号を`np.argsort`で求め、いちばんめずらしい5個の名前と、`lik`の値を表示してください。


In [ ]:
# TODO: lik の小さい順に、5個の名前と値を表示する

# 解答例④：いちばんめずらしい5個
rare = np.argsort(lik)[:5]
print(indexes[rare])
print(lik[rare].round(0))


いちばんめずらしい5個は、`lik`が大きくマイナスの銀河です。どんな銀河かは、次の著者のセルで画像にして見ます。


### 外れ値のセル

著者のセル16は、`lik`の小さい順と大きい順に、25個ずつカラー画像を並べ、ファイルに保存します。


In [ ]:
# 著者のセル16をそのまま写したもの
lik_idx=indexes[np.argsort(lik)]
P=make_image(lik_idx[0:25],(5,5))
P.tight_layout()
P.savefig("most-rare.png")
P=make_image(np.flip(lik_idx)[0:25],(5,5))
P.tight_layout()
P.savefig("least-rare.png")


In [ ]:
plt.show()


1枚目（めずらしい25個）には、とても明るく青い点（星が写りこんだのかもしれません）、2つ以上が1枚に写ったもの、大きく広がった明るい銀河が並びます。2枚目（ありふれた25個）は、小さく暗い銀河ばかりです。外れ値を探すと、めずらしい銀河だけでなく、データの汚れ（写りこみ・重なり）も見つかります。


---


## 6. 似た銀河は、どう集まるか？


めずらしい銀河は見ました。では、似た形の銀河どうしは、どう集まるでしょうか。著者は、固有銀河の得点`scores`で、銀河をグループに分けています。


### Affinity Propagation

著者が使っているのは、Affinity Propagation（アフィニティ・プロパゲーション）という方法です。k-meansと違って、グループの数を先に決めません。各点が「自分の代表にふさわしい点」を互いに推しあい、代表と、そこに集まるグループが決まります。


### クラスタリングのセル

著者のセル33を、そのまま実行します。計算に数秒かかります。


In [ ]:
# 著者のセル33をそのまま写したもの
from sklearn.cluster import AffinityPropagation
clust=AffinityPropagation(damping=0.9,convergence_iter=100,max_iter=1000)
clust=clust.fit(scores)


In [ ]:
print(len(np.unique(clust.labels_)), 'グループ')


百四十数個のグループができました。この数は、実行するたびに1つ2つ揺れることがあります。著者のセルは、乱数のシード（`random_state`）を決めていないからです。もう一度実行して、確かめてみてください。


### 変えてみる：乱数のシードを決める

- 役割：`random_state=0`を足して、同じ設定でもう一度グループに分ける
- 入力：`scores`
- 出力：グループの数

前回の`KMeans(random_state=0)`と同じく、乱数のシードを決めると、何度実行しても同じ結果になります。ここから先は、この`clust`を使います。


In [ ]:
from sklearn.cluster import AffinityPropagation
clust = AffinityPropagation(damping=0.9, convergence_iter=100,
                            max_iter=1000, random_state=0).fit(scores)
print(len(np.unique(clust.labels_)), 'グループ')


146グループです。


### グループの要約のセル

著者のセル34の前半は、グループの大きさを要約して表示します（後半は、グループごとの画像をファイルに保存する長い処理なので、写していません）。


In [ ]:
# 著者のセル34をそのまま写したもの
import os

l=clust.labels_
G=pd.DataFrame(dict(l=l)).groupby("l").l.count()
df=pd.DataFrame(dict(l=G.index,cnt=G.values))
df=df.sort_values("cnt",ascending=False)
df=df.head(100)

print("labels",len(np.unique(l)))
print("min",np.min(G))
print("max",np.max(G))
print("mean",np.median(G))
print("median",np.median(G))


グループは146個、いちばん小さいグループは1個、いちばん大きいグループは143個です。そして、`mean`（平均）と`median`（中央値）は、どちらも1.0と表示されています。グループの大きさの平均が1個で、本当によいのでしょうか。


### TODO⑤：要約の表示を確かめる

著者のセル34の、`print("mean",…)`の行をよく読んでください。何を計算しているでしょうか。グループの大きさ`G`の平均を`np.mean`で、中央値を`np.median`で、それぞれ表示してください。


In [ ]:
# TODO: グループの大きさの平均と中央値を表示する

# 解答例⑤：グループの大きさの平均と中央値
print('mean', round(np.mean(G), 2))
print('median', np.median(G))


`print("mean",np.median(G))`は、平均と書きながら中央値を計算していました。本当の平均は8.37で、中央値は1です。見出しと中身が食い違っていたのです。

平均と中央値がこれほど違うのは、大きさ1のグループがとても多いからです。数えてみます。


In [ ]:
print((G == 1).sum(), '個のグループが、銀河1個だけ')
big = G[G >= 10]
print(len(big), '個のグループが10個以上で、合わせて', big.sum(), '個')


146グループのうち90グループは、銀河1個だけのグループです。10個以上の銀河が入ったグループは25個で、合わせて1,012個の銀河が入っています。1,222個の銀河の多くは、25個ほどの大きなグループに集まり、残りはどこにも似ていない1個ずつのグループになっていました。


### 変えてみる：いちばん大きいグループを見る

- 役割：いちばん大きいグループの銀河の名前を取り出し、先頭の30個のカラー画像を並べる
- 入力：`clust.labels_`、`G`、`indexes`
- 出力：画像の並び

`G.idxmax()`は、`G`の値がいちばん大きい所の番号（グループの番号）を返します。


In [ ]:
top = indexes[clust.labels_ == G.idxmax()]
make_image(top[:30], (5, 6))
plt.show()


いちばん大きいグループは、小さく丸い、中心の明るい銀河です。主成分分析の得点が近い銀河は、目で見ても形が似ています。


### 答え

論文の主張は、手元の1,222個でも確かめられました。

- 前処理（明るさのクリップと、向きをそろえる）のあと、10個の固有銀河で、形の散らばりの96%を表せた。論文の約12個に近い
- 3個で9割を表せ、第1主成分だけで寄与率は0.763
- その得点で、似た形の銀河のグループに分けられた。1,012個の銀河が、10個以上の25グループに入った

再現するときに気をつける所も、見つかりました。

- ファイルの並び順（`glob`）と、乱数のシードで、結果が揺れる。並べ方とシードを決めておく
- 著者の関数は、渡した配列をその場で書きかえることがある。写しを渡す
- 表示の見出しと中身が食い違うことがある（`mean`と書いて中央値）。数は、自分で計算して確かめる


まだ言えないこと

- 前処理の効果：向きをそろえる段がなかったら、固有銀河の数はどう変わるか（課題）
- 1万個での再現：論文は約1万個。今日はその一部の1,222個で確かめた
- グループの数は妥当か：146は多すぎないか。分け方の良し悪しを、数で測るには


---


## 目標に答えられたか


- 今日の目標は「論文の主張を、著者が公開したコードを読んで動かし、確かめられるか」でした。何個の固有銀河で何%を表せましたか？ 論文の数と比べてどうでしたか？
- 1節で、著者のノートを、どうやって地図にしましたか？`.ipynb`の中身はどんな形でしたか？
- 2節で、.npyとPNGは、何が違いましたか？`glob`の順番について、何に気をつけましたか？
- 3節で、向きは、何に主成分分析をかけて測っていましたか？ なぜ`.copy()`を渡しましたか？
- 5節で、めずらしい銀河には、どんなものが入っていましたか？
- 6節で、著者の要約の表示には、何が起きていましたか？


---


## 今日の要点


- 今日、持ち帰る1文：他人のコードは、見出しで地図を作り、関数を作るブロックと処理のブロックに分けて読む。動かして数を出し直し、並び順・乱数のシード・書きかえ・表示の食い違いのように、結果が揺れたり誤ったりする所を確かめてから、論文の主張と比べる
- `.ipynb`はJSON。辞書とリストの入れ子として読める
- 形式ごとに読み方が違う：.npyは`np.load`、PNGは`plt.imread`。たくさんのファイルは`glob`で集める
- パイプライン（pipeline）：読み込み → 明るさのクリップ → 向きをそろえる（画素の座標に主成分分析）→ 主成分分析（固有銀河）→ 外れ値 → クラスタリング
- Python：`json.load`、`glob`、`np.load`、`plt.imread`、`os.chdir`、`!`で始まる行、`.copy()`、`random_state`


---


## 次回へ


今日のクラスタリングは、146ものグループに分かれました。前回の国のグループ分けでも、3つというグループの数は、こちらで決めた数でした。

次回の問いは、その分け方は妥当か、グループの数をいくつにすべきかを、根拠をもって言えるか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

著者の前処理`normalise_all`は、「明るさのクリップ」と「向きをそろえる」の2段の処理をしていました。今度は向きをそろえる段を外し、明るさのクリップだけをした表`clip_only`を作って、`PCA(.96)`をかけてください。主成分の数と、上位3つの寄与率を表示しましょう。

`np.quantile(raw_data, [0, .99])`で、クリップの下と上の値が求まります（`normalise_all`の中と同じ）。固有銀河を9個並べるには、`make_image_pre(pca_r.components_[:9].copy(), (3, 3), '向きをそろえない')`とします。


In [ ]:
L, U = np.quantile(raw_data, [0, .99])
clip_only = np.clip(raw_data, L, U)        # 向きをそろえない
pca_r = PCA(.96).fit(clip_only)
print(pca_r.n_components_, pca_r.explained_variance_ratio_[:3].round(3))
make_image_pre(pca_r.components_[:9].copy(), (3, 3), '向きをそろえない')
make_image_pre(pca.components_[:9].copy(), (3, 3), '向きをそろえる')
plt.show()


<details><summary>詰まったら</summary>

`L, U = np.quantile(raw_data, [0, .99])` → `clip_only = np.clip(raw_data, L, U)`。
`pca_r = PCA(.96).fit(clip_only)`で、主成分の数は`pca_r.n_components_`です。

</details>


### 応用②（判断）

応用①の、向きをそろえないときの主成分の数（寄与率96%まで）はいくつですか。整数で答えてください。


In [ ]:
print('答え:', pca_r.n_components_)


<details><summary>詰まったら</summary>

`pca_r.n_components_`を表示します。4節（向きをそろえたとき）は10個でした。

</details>


### 応用③（解釈）

向きをそろえたとき（4節）と、そろえないとき（応用①）で、必要な固有銀河の数と、固有銀河の形はどう変わりましたか。「向きをそろえる」という前処理の1段が、この論文の主張にとってなぜ要るのかを、研究室の先輩に向けて3行程度で書いてください。


（模範例）

向きをそろえたときは10個の固有銀河で96%を表せましたが、そろえないと14個が要りました。そろえないときの固有銀河には、同じ伸びた形を、いろいろな向きで表すための成分が混ざります。

向きが違うだけの同じ形を、別々の成分で表すことになるので、そのぶん固有銀河の数が増えます。向きをそろえると、同じ96%に要る数が14個から10個に減ります。

論文の「約12個」という少なさは、向きをそろえる段を含めた数です。論文の数と比べるときは、この1段を省かずに再現する必要があります。


---


## 発展（任意）


### 脳の画像を、区画に分ける（Abrahamほか2014）

医用画像にも、同じ「読み込み → 前処理 → 次元の扱い → クラスタリング」のパイプラインがあります。Abrahamほか（2014, Frontiers in Neuroinformatics「Machine learning for neuroimaging with scikit-learn」）は、安静にしている人の脳のfMRI（脳の活動の時間変化を撮るMRI）を、似た動きをする場所ごとの区画に分けました。

論文のコードは古い版の道具で書かれていて今は動かないので、同じ手順を今の道具（nilearnとscikit-learn）で書き直した版で、論文の最後の図（脳の区画分け）を再現します。

- 形式：脳の画像はNIfTI（`.nii.gz`）という医用画像の形式。縦×横×高さ×時間の4次元の配列です
- データ：nilearnが公開データから1人分（約44MB、20秒ほど）を取ってきます


In [ ]:
!pip install -q nilearn
from nilearn import datasets, plotting
from nilearn.maskers import NiftiMasker
import nibabel as nib

adhd = datasets.fetch_adhd(n_subjects=1)        # 1人分を取ってくる
img = nib.load(adhd.func[0])                    # NIfTI を読む
print(img.shape)                                # 縦・横・高さ・時間


61×73×61の立体（1つの画素は3mm角）を、2秒ごとに176回撮っています。


In [ ]:
# 脳の中だけを取り出し（マスク）、少しぼかし、場所ごとに標準化する
masker = NiftiMasker(mask_strategy='epi', smoothing_fwhm=6,
                     standardize='zscore_sample')
X = masker.fit_transform(img)                   # 時間 × 脳の場所
print(X.shape)


176時点 × 62,546か所の表になりました。1列が、脳の1か所（ボクセル）の、時間の動きです。


In [ ]:
from sklearn.feature_extraction.image import grid_to_graph
from sklearn.cluster import FeatureAgglomeration

mask = masker.mask_img_.get_fdata().astype(bool)
conn = grid_to_graph(*mask.shape, mask=mask)  # となりだけつなぐ
ward = FeatureAgglomeration(n_clusters=100, connectivity=conn,
                            linkage='ward').fit(X)
labels = masker.inverse_transform(ward.labels_ + 1)   # 脳の形に戻す
plotting.plot_roi(labels, title='Ward 法で100の区画',
                  display_mode='xz', cut_coords=(0, 0))
plotting.show()


読み方：色の1つ1つが、時間の動きが似ている場所をまとめた区画です。Ward法に「となり合う場所だけをつなぐ」という条件（`connectivity`）を付けたので、どの区画もひとつながりの塊になっています。

列（場所）をグループに分けているので、使っているのは`FeatureAgglomeration`（列をまとめる版の階層クラスタリング）です。第8回で遺伝子のまとまりを見た階層クラスタリングと、同じ仲間の方法です。

試すなら、`n_clusters`を20や500に変えて、区画の細かさがどう変わるか見てみましょう。また、つながりの条件なしのk-means（`MiniBatchKMeans(n_clusters=100).fit(X.T)`）で分けると、区画がばらばらに散らばることも確かめられます。
